In [1]:
import pdfplumber
import os
import re
import pandas as pd

In [2]:
def parse_pdf(file):
    INVOICE_NUMBER_PATT = re.compile(r'(CD2026[A-Z0-9-]+)', re.IGNORECASE)
    TOTAL_AMOUNT_PATT = re.compile(r'(?:Invoice\s*amount)\s*([\d\s.,]*)', re.IGNORECASE)
    HS_CODE_PATT = re.compile(r'\b([\d]{8,10})\b', re.IGNORECASE)
    WEIGHT_PATT = re.compile(r'([\d.,]+)\s(?:kg)')
    page_text = ''

    data = {
        "Invoice number": None,
        "Total amount": None,
        "Items": []
    }
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            page_text += page.extract_text(keep_blank_chars = True) or '-- TEXT NOT FOUND --'
    
    invoice_found = INVOICE_NUMBER_PATT.search(page_text)
    total_found = TOTAL_AMOUNT_PATT.search(page_text)

    if invoice_found: data['Invoice number'] = invoice_found.group(1).strip()
    if total_found:
        total_string = total_found.group(1).strip()
        clean_string = re.sub(r'[^\d.]','', total_string)
        data['Total amount'] = float(clean_string)

    for line in page_text.split('\n'):
        if re.match(r'\b[\d]{8,10}\b', line):
            hs = HS_CODE_PATT.search(line)
            weight = WEIGHT_PATT.search(line)
            weight_str = weight.group(1).strip()
            clean_weight = re.sub(r'[^\d.]', '', weight_str)
            data['Items'].append({'HS': hs.group(1).strip(), 'weight': float(clean_weight)})

    return data


In [3]:
path = 'files'

In [4]:
result = []

In [5]:
for file_name in os.listdir(path):
    if file_name.endswith('.pdf'):
        file_path = os.path.join(path, file_name)
        try:
            fetched_data = parse_pdf(file_path)

            if fetched_data['Items']:
                for item in fetched_data['Items']:
                    result.append({
                        'Invoice': fetched_data['Invoice number'],
                        'HS code': item['HS'],
                        'Weight': item['weight'],
                        'Amount': fetched_data['Total amount']
                    })
            else:
                for item in fetched_data['Items']:
                    result.append({
                        'Invoice': fetched_data['Invoice number'],
                        'HS code': '-- not found --',
                        'Weight': '-- not found --',
                        'Amount': fetched_data['Total amount']
                    })
        except Exception as e:
            print(f'Error reading file {file_name}: {e}')

Error reading file CD202606708.pdf: could not convert string to float: ''
Error reading file Customs invoice C395144.pdf: could not convert string to float: ''
Error reading file Customs invoice C395150.pdf: could not convert string to float: ''


In [6]:
df = pd.DataFrame(result)
df.to_excel('parsed_data.xlsx', index=False)
print('Parsing done')

Parsing done
